# 자습 10. 시연을 학습 데이터로 만들기

> **선택 자습 — 본수업 8시간에 포함하지 않습니다.** 코드 셀은 소형 합성 예제이며 GPU 학습·다운로드·로봇 구동을 자동 실행하지 않습니다. 실제 작업 명령은 설명의 터미널 예시로 제공하며 준비물을 먼저 갖춥니다. TODO는 직접 작성하고 아래 정답·해설로 검증하세요. [전체 자습 경로](selfstudy/README.md)


## A. 강사 설명 — 아직 코드를 실행하지 않습니다

**설명 자료:** [강의자료_5일차_v2 설명 70~70쪽](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_selfstudy_10) → [실습 시작 71쪽](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_optional10_start).

강사는 아래 순서로 이론과 노트북 사용 방법을 설명합니다. 수강생은 입력과 예상 결과를 말로 예측합니다.

1. 무엇을 만들며 왜 필요한가요?
2. 수집 순서
3. LeRobotDataset v3에 저장할 때
4. TODO 1 — 프레임 대신 수집 그룹으로 분리
5. TODO 2 — episode 끝의 행동 묶음

노트북에서 `10_selfstudy_demonstration_data`의 입력·처리·출력을 보여 주고, 실행 위치와 수정할 항목을 짚습니다. 예제 결과와 실패 사례를 설명한 뒤, 아래 **B. 수강생 실습 시작**에서 실행을 시작하도록 안내합니다.

**완료 기준:** 노트북의 TODO·확인 질문과 결과 기록 완료

**선택 자습:** 본수업 필수 완료 범위 밖입니다. 선택 수업으로 운영할 때에도 설명 → 실습 → 확인 순서를 따릅니다.


## B. 수강생 실습 시작

강사의 설명과 실습 방법 시연이 끝나면 여기부터 진행합니다. 아래 이론 문단은 실습 중 참고할 설명입니다.

1. 이 노트북의 커널을 새로 시작하고 첫 준비·import 셀부터 실행합니다. 다른 단원의 `common` 모듈이 남아 있지 않도록 노트북별 새 커널을 사용합니다.
2. 예제 셀의 결과를 예측한 뒤 한 셀씩 실행합니다. 터미널 명령은 표시된 호스트/컨테이너 터미널에 입력합니다.
3. 노트북에 표시된 TODO 또는 확인 질문을 작성하고 실행 결과와 비교합니다.
4. 실제 입력 실행·결과 저장을 마친 뒤 C의 완료 기록을 작성합니다. 참고 정답은 자신의 시도 후 강사 안내에 따라 비교합니다.

**실행 범위:** 10_selfstudy_demonstration_data


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| feature key/shape/dtype 불일치 | 카메라 이름과 영상·상태·행동 규격을 계약에 맞춥니다. 의미 확인 없이 reshape하지 않습니다. | 저장 후 재로딩한 규격 일치 |
| 데이터 누락·video decode 오류 | episode 저장/finalize, 영상 파일·시간 정보와 backend를 확인합니다. | 한 episode의 영상·상태·행동 대응 |
| 분할/경계 TODO 실패, 평가가 비정상적으로 좋음 | 수집 그룹 겹침, train 외 통계 사용, chunk 경계의 padding/mask를 검사합니다. | 그룹 분리와 짧은 episode 경계 검사 통과 |

**예시에 없는 오류를 확인할 곳:** [VLA 단계별 오류 사전](selfstudy/TROUBLESHOOTING.md)의 10 절과 [준비 안내](selfstudy/README.md). 훈련은 터미널 stdout/stderr와 checkpoint 설정, 재생은 지정 output의 결과 파일을 봅니다.

VLA 훈련·저장 관측 재생에는 noVNC 창이 열리지 않습니다. 본수업 GUI rollout을 다시 실행한 경우에만 FINISHED 후 MuJoCo 창을 닫습니다.

[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 무엇을 만들며 왜 필요한가요?

영상 파일이 있다는 것만으로 로봇이 어떤 행동을 따라 배워야 하는지는 알 수 없습니다. 한 시연 episode에는 시간 순서대로 **관측, 로봇 상태, 실제 적용 명령, 작업 지시** 가 들어갑니다. 수집 시점, 실행 지연, 성공 여부와 조작자도 별도 기록합니다.

**완료 결과:** 데이터 계약·episode 목록·분할 규칙·작은 데이터 로딩 검사. 설계·합성 실습은 90분, 실제 수집은 별도입니다. 09의 계약이 선행 조건입니다.

![시연 데이터의 시간 정렬과 분할](../교재그림/vla-selfstudy-data.svg)

예를 들어 20Hz라면 표본 간 목표 간격은 0.05초입니다. 이 숫자는 예시입니다. 실제 수집 timestamp로 간격을 확인하고, 카메라가 10Hz인데 값을 복제해 20Hz처럼 쓰는 경우 중복·나이를 명시해야 합니다. 미래 프레임으로 과거 행동을 학습시키는 정렬 오류도 피합니다.


## 수집 순서

<!-- procedure:lesson-46-986cd2:start -->
![시연을 저장하고 다시 검증](../교재그림/절차/lesson-46.svg)

인접 프레임을 무작위로 나누면 정보가 새어 나갈 수 있습니다. 수집 그룹과 episode 경계를 보존하세요.
<!-- procedure:lesson-46-986cd2:end -->


1. 로봇별 센서·행동 API와 보정을 확인합니다. 09의 성분·단위·좌표계 계약을 파일로 고정합니다.
2. 조작자로부터 행동을 받기 직전의 관측을 읽고, 시간·episode·프레임 번호를 기록합니다.
3. 제한·보정을 거친 뒤 제어기에 실제 전달한 명령을 학습용 `action`으로 기록합니다. 원래 요청과 실제 전달값이 다르면 둘 다 감사 기록에 남깁니다.
4. 다음 관측을 수집하며 반복하고, 성공/실패/중단과 원인을 episode 단위로 기록합니다.
5. 사람이 몇 개 영상을 확인하여 시간 어긋남·가려짐·카메라 순서·명령 부호를 검사합니다.

수집 어댑터는 로봇마다 다르므로 이 자료에 가짜 장치 드라이버를 넣지 않습니다. LIBERO에서는 `env.step`에 전달한 행동과 그 직전 관측을 저장하도록 별도 수집기를 구현해야 합니다. 본수업 `run_episode.py`는 추론 평가용이며 이 전체 데이터 수집 기능을 제공하지 않습니다.

반복 성공 시연을 먼저 확보하고 초기 배치·조명·물체를 계획적으로 바꿉니다. 실패 시연을 성공 시연과 섞어 동일한 모방 목표로 학습시키지 않습니다. 실패는 평가와 추가 수집 계획에도 유용합니다. 정답이 잘못된 데이터가 많아지는 것은 데이터 다양성과 다릅니다.


## LeRobotDataset v3에 저장할 때

0.4.4의 `LeRobotDataset.create → add_frame → save_episode → finalize` 흐름을 사용합니다. v3의 episode 경계는 메타데이터로 관리되며 MP4 하나가 episode 하나라고 가정하지 않습니다. `finalize`를 호출해 저장을 닫은 다음 새 프로세스에서 다시 읽습니다.

아래는 **설명용 수집 루프 골격** 입니다. `read_observation`, `teleop_action`, `send_action`은 자신의 장비 어댑터로 구현해야 하며 바로 실행 가능한 로봇 프로그램이 아닙니다. 실습 코드 셀에서는 장치나 파일을 만들지 않습니다.

```python
# 준비된 별도 수집 환경에서만 적용할 구조 예시
from lerobot.datasets.lerobot_dataset import LeRobotDataset
features = {
    'observation.state': {'dtype': 'float32', 'shape': (6,),
                          'names': ['j1','j2','j3','j4','j5','gripper']},
    'action': {'dtype': 'float32', 'shape': (6,),
               'names': ['j1','j2','j3','j4','j5','gripper']},
    'observation.images.front': {'dtype': 'video', 'shape': (480,640,3),
                                  'names': ['height','width','channels']},
}
# root는 아직 존재하지 않는 새 데이터셋 경로
# dataset = LeRobotDataset.create(repo_id='local/pickplace_train',
#     root=TRAIN_ROOT, fps=20, features=features, robot_type=ROBOT_TYPE,
#     vcodec='libx264', video_backend='pyav')
# try:
#     for episode in demonstrations:
#         for observation, applied_action in episode:
#             dataset.add_frame({'observation.state': observation['state'],
#                 'observation.images.front': observation['front'],
#                 'action': applied_action, 'task': 'Place the block in the tray.'})
#         dataset.save_episode()
# finally:
#     dataset.finalize()
```

shape6·20Hz·영상480×640은 **가상의 관절 로봇 예시** 입니다. LIBERO 상태8·행동7과 혼용하지 않습니다. HWC uint8 RGB·float32 벡터·성분 이름을 검사합니다. 예시 codec의 사용 가능 여부는 자신의 FFmpeg 빌드에서 확인합니다. root를 미리 만들어 놓으면 `create`가 기존 경로 오류를 낼 수 있으므로 부모 폴더만 준비합니다.


## TODO 1 — 프레임 대신 수집 그룹으로 분리

아래 합성 목록에서 `day-A`는 train, `day-B`는 validation, `day-C`는 test로 배정하세요. 같은 날짜의 episode가 여러 집합에 들어가지 않게 합니다. 실제 프로젝트에서는 날짜뿐 아니라 조작자·물체·배치 등 시험하려는 일반화 기준으로 group을 정의합니다.


In [ ]:
episodes = [
    {'id': 0, 'group': 'day-A'}, {'id': 1, 'group': 'day-A'},
    {'id': 2, 'group': 'day-B'}, {'id': 3, 'group': 'day-C'},
]
def assign_split(group):
    # TODO: group별 train / validation / test 문자열 반환
    return None
splits = {row['id']: assign_split(row['group']) for row in episodes}
if None in splits.values():
    print('TODO 미완료: assign_split을 작성하세요.')
else:
    assert splits == {0:'train', 1:'train', 2:'validation', 3:'test'}
    print('그룹 분리 검증 통과:', splits)

### 정답과 해설

```python
def assign_split(group):
    return {'day-A':'train', 'day-B':'validation', 'day-C':'test'}[group]
```

옆 프레임은 장면이 매우 비슷하므로 프레임 무작위 분할은 평가에 학습 장면이 새어 들어가게 합니다. episode만 나눠도 동일 배치에서 연속 수집한 두 시연이 양쪽에 있으면 더 엄격한 일반화 검증에는 부족할 수 있습니다. 수집 그룹을 먼저 나누고 그 안의 episode를 배정합니다. 이 합성 4개로 실제 성능을 검증했다고 주장할 수는 없습니다.

**정규화 통계도 train에서만 계산합니다.** validation/test까지 포함한 데이터셋의 통계로 학습하면 정보가 새어 들어갑니다. 처음에는 train·validation·test를 별도 dataset root로 만들고 train의 stats로만 전처리를 학습·저장하세요. 단순 `--dataset.episodes` 필터가 모든 기존 통계를 다시 계산한다고 가정하지 않습니다.


## TODO 2 — episode 끝의 행동 묶음

한 episode의 행동이 `[10, 11, 12, 13]`일 때 마지막 인덱스3에서 길이3의 묶음을 만들어 보세요. 다음 episode의 첫 행동을 가져오면 안 됩니다. 아래 과제는 **끝 값을 반복하고 반복 위치를 padding으로 표시** 하는 방법을 연습합니다. 실제 학습 loader에서는 저장된 episode 경계와 `action_is_pad` 처리가 올바른지 확인합니다.


In [ ]:
def padded_chunk(actions, start, length):
    # TODO: (행동 리스트, padding 여부 리스트) 반환
    return None
answer = padded_chunk([10,11,12,13], 3, 3)
if answer is None:
    print('TODO 미완료: padded_chunk를 작성하세요.')
else:
    assert answer == ([13,13,13], [False,True,True])
    assert padded_chunk([10,11,12,13], 1, 2) == ([11,12], [False,False])
    print('episode 경계 검증 통과')

### 정답과 해설

```python
def padded_chunk(actions, start, length):
    if not actions or not 0 <= start < len(actions) or length < 1:
        raise ValueError('잘못된 묶음 요청')
    indices = range(start, start + length)
    values = [actions[min(i, len(actions)-1)] for i in indices]
    mask = [i >= len(actions) for i in indices]
    return values, mask
```

반복한 13은 실제 새 행동 표본이 아니므로 mask=True입니다. mask 없이 학습하면 episode 뒤의 가짜 정지 행동을 과도하게 학습할 수 있습니다. 미래 행동 길이와 실제 실행 길이는 다릅니다. 학습에서 50개를 예측하도록 구성해도 배포에서 앞 10개만 적용할 수 있습니다.

**실제 데이터 완료 기준:** 임의 episode 3개 재생, 키·shape·dtype·NaN 검사, timestamp 단조성·간격 검사, 관측/행동 정렬 확인, train/test group 교집합 없음, 저장 후 새 프로세스 재로딩 성공. 수치 검사만으로 정답 행동의 품질까지 보장되지 않으므로 시연 영상을 함께 봅니다.

다음: [11 로컬 미세조정](11_selfstudy_local_training.ipynb). 형식 참고: [공식 LeRobotDataset v3](https://huggingface.co/docs/lerobot/v0.4.3/en/lerobot-dataset-v3), 정확한 API 기준: [0.4.4 데이터셋 소스](https://github.com/huggingface/lerobot/blob/v0.4.4/src/lerobot/datasets/lerobot_dataset.py).


## C. 실습 완료 확인

노트북의 TODO·확인 질문과 결과 기록 완료

아래 항목을 본인의 결과로 채우고 노트북을 저장합니다.

| 항목 | 내 기록 |
|---|---|
| 실행한 입력·조건 | 작성 |
| 수정한 코드 또는 답안 | 작성 |
| 결과 파일·검증 출력 | 작성 |
| 예상과 달랐던 결과·이유 | 작성 |
| 미완료 항목·오류·다음 조치 | 작성 |

강사와 완료 기준을 확인한 후 다음 단원의 설명으로 이동합니다.
